In [1]:
%pip install pyproj requests pandas IPython datetime display python-dotenv

Note: you may need to restart the kernel to use updated packages.


# 📢 경기도 소식 및 행사 현황 API 연동 실습

이 문서는 경기데이터드림의 오픈 API를 활용하여 경기도 지역의 행사 및 소식 데이터를 연동하는 파이썬 코드를 정리한 내용입니다.

## 1. 경기도 소식 및 행사 현황 API 호출

경기데이터드림의 '경기도 소식 및 행사 현황(GGNEWSSTUS)' 서비스를 사용하여 경기도의 최신 소식, 축제, 행사 데이터를 실시간으로 가져옵니다.

### 1.1 기본 API 호출 및 데이터 프레임 변환

API 호출 시 응답 결과는 JSON 형태로 반환됩니다. 아래는 발급받은 인증키를 사용해 원본 데이터를 호출하고, 응답받은 데이터를 파싱(Parsing)하여 직관적인 Pandas 데이터 프레임 표 형태로 변환하여 출력하는 기본 코드입니다.

In [21]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv, find_dotenv
from IPython.display import display

# 1. 환경변수(.env) 로드
load_dotenv(find_dotenv())
EVENT_API_KEY = os.getenv("GYEONGGI_EVENT_API_KEY")

# 2. API 엔드포인트 설정 (명세서 기준)
API_NAME = "GGNEWSSTUS"
API_URL = f"https://openapi.gg.go.kr/{API_NAME}"

# 3. 기본 인자 세팅
params = {
    "KEY": EVENT_API_KEY,  
    "Type": "json",        
    "pIndex": 1,           
    "pSize": 1000           
}

# 4. API 호출 및 데이터 전처리
try:
    print(f"'{API_NAME}' API 호출 중...")
    response = requests.get(API_URL, params=params)
    
    if response.status_code == 200:
        data = response.json()
        
        # 5. 파싱 구조: data['GGNEWSSTUS'][1]['row']
        if API_NAME in data:
            rows = data[API_NAME][1].get("row", [])
            
            # 6. Pandas DataFrame으로 변환
            event_df = pd.DataFrame(rows)
            
            print(f"\n✅ 경기도 소식 및 행사 데이터 수집 성공! (총 {len(event_df)}건)")
            display(event_df.head()) # 상위 5개 데이터 표 형태로 출력
            
        else:
            print("데이터를 찾을 수 없습니다. API 명칭이나 키를 다시 확인해 주세요.")
            print("원본 응답:", data)
            
    else:
        print(f"HTTP 오류 발생: {response.status_code}")
        print(response.text)

except Exception as e:
    print(f"API 연동 중 오류가 발생했습니다: {e}")

'GGNEWSSTUS' API 호출 중...

✅ 경기도 소식 및 행사 데이터 수집 성공! (총 897건)


,INST_NM,TITLE,CATEGORY_NM,URL,IMAGE_URL,BEGIN_DE,END_DE,WRITNG_DE
0,경기문화재단,[용인농촌테마파크] 토요상시체험 진행 (2/21),일반소식,https://ggc.ggcf.kr/news/view/6998fc24325a86b1...,https://ggc.ggcf.kr/public/images/thumDefaultI...,2026-02-21,2026-02-21,2026-02-21
1,경기문화재단,[경기도박물관] 2026 문화동호회 신규회원 모집 (2/22~3/5),일반소식,https://ggc.ggcf.kr/news/view/6998fc8f325a86b1...,https://ggc.ggcf.kr/public/images/thumDefaultI...,2026-02-22,2026-03-05,2026-02-21
2,경기문화재단,[처인성역사교육관] 겨울방학 교육프로그램 신청 (2/9~2/28),일반소식,https://ggc.ggcf.kr/news/view/698bd0a4325a86b1...,https://ggc.ggcf.kr/public/images/thumDefaultI...,2026-02-09,2026-02-28,2026-02-11
3,경기문화재단,[경기도박물관] 특별전_성파선예: 성파스님의 예술세계 (2/10~5/31),일반소식,https://ggc.ggcf.kr/news/view/698bd02d325a86b1...,https://ggc.ggcf.kr/public/images/thumDefaultI...,2026-02-10,2026-05-30,2026-02-11
4,경기문화재단,[한국민속촌] 설날 세시행사 (2/14~3/3),일반소식,https://ggc.ggcf.kr/news/view/698bcf94325a86b1...,https://ggc.ggcf.kr/public/images/thumDefaultI...,2026-02-14,2026-03-03,2026-02-11


In [20]:
# API 응답 결과 데이터(data)가 있다고 가정할 때
if "GGNEWSSTUS" in data:
    total_count = data["GGNEWSSTUS"][0]["head"][0].get("list_total_count")
    print(f"📢 서버에 저장된 전체 데이터 개수: {total_count}개")

📢 서버에 저장된 전체 데이터 개수: 897개


## 2. LLM 활용

openai api 키 불러와서 행사명에서 자동으로 도시명을 연결

🛠️ Step 1. 사전 준비 (API 키 설정), openAI 라이브러리 설치
먼저 .env 파일에 OpenAI API 키가 등록되어 있어야 합니다.
추가한 후,
파이썬 환경에 OpenAI 라이브러리가 설치합니다.

In [14]:
!pip install openai python-dotenv

  Using cached anyio-4.12.1-py3-none-any.whl.metadata (4.3 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached jiter-0.13.0-cp311-cp311-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached pydantic_core-2.41.5-cp311-cp311-win_amd64.whl.metadata (7.4 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 3.9 MB

In [22]:
import os
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_city_via_llm(title):
    """
    LLM을 사용하여 행사 제목에서 경기도의 시/군 명칭을 추출합니다.
    """
    prompt = f"""
    아래의 행사 제목을 보고, 이 행사가 경기도의 어떤 '시'나 '군'에서 열리는지 찾아줘.
    결과는 반드시 '용인시', '수원시', '가평군'과 같은 형식으로 시/군 이름만 딱 한 단어로 답변해.
    만약 제목에서 유추할 수 없다면 '기타'라고 답변해.

    행사 제목: {title}
    도시 이름:
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini", # 비용 절감을 위해 mini 모델 추천
            messages=[{"role": "user", "content": prompt}],
            max_tokens=10,
            temperature=0
        )
        city = response.choices[0].message.content.strip()
        return city
    except Exception as e:
        print(f"Error for {title}: {e}")
        return "기타"

# 1. API로 불러온 데이터프레임 (예: event_df)
# 샘플 테스트를 위해 상위 10개만 먼저 해보는 것을 추천합니다.
test_df = event_df.head(10).copy()

print("LLM이 도시를 분석 중입니다...")
test_df['CITY'] = test_df['TITLE'].apply(get_city_via_llm)

# 2. 결과 확인
display(test_df[['CITY', 'TITLE', 'INST_NM']])

LLM이 도시를 분석 중입니다...
Error for [용인농촌테마파크] 토요상시체험 진행 (2/21): Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
Error for [경기도박물관] 2026 문화동호회 신규회원 모집 (2/22~3/5): Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
Error for [처인성역사교육관] 겨울방학 교육프로그램 신청 (2/9~2/28): Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
Error for [경기도박물관] 특별전_성파선예: 성파스님의 예술세계 (2/10~5/31): Error code: 401 - {'error': {'message': 'Incor

,CITY,TITLE,INST_NM
0,기타,[용인농촌테마파크] 토요상시체험 진행 (2/21),경기문화재단
1,기타,[경기도박물관] 2026 문화동호회 신규회원 모집 (2/22~3/5),경기문화재단
2,기타,[처인성역사교육관] 겨울방학 교육프로그램 신청 (2/9~2/28),경기문화재단
3,기타,[경기도박물관] 특별전_성파선예: 성파스님의 예술세계 (2/10~5/31),경기문화재단
4,기타,[한국민속촌] 설날 세시행사 (2/14~3/3),경기문화재단
5,기타,2026 겨울방학 특강(곤충) 체험객 모집 (~2/9),경기문화재단
6,기타,[용인농촌테마파크] 토요상시체험 진행 (2/7),경기문화재단
7,기타,[경기도박물관] 경기 트레저 헌팅,경기문화재단
8,기타,[용인시기후변화체험교육센터] 2026년 겨울방학 특별프로그램 (2/5~2/27),경기문화재단
9,기타,[용인문화원] 제14회 느린 손바느질 이야기 (2/4~2/7),경기문화재단


🚀 Step 3. 하이브리드 방식 추천 (비용 및 속도 최적화)
모든 데이터(수백 건)를 LLM으로 돌리면 비용이 발생하고 속도가 느려집니다. "1차는 키워드 매칭, 2차는 LLM 추론" 방식을 추천합니다.

In [23]:
# 1차: 명확한 키워드(용인, 수원 등)는 코드로 즉시 처리
def fast_extract(title):
    if "용인" in title: return "용인시"
    if "수원" in title: return "수원시"
    return None # 매칭 안 되면 LLM으로 넘김

event_df['CITY'] = event_df['TITLE'].apply(fast_extract)

# 2차: CITY가 아직 None인 행만 LLM 호출
mask = event_df['CITY'].isna()
event_df.loc[mask, 'CITY'] = event_df.loc[mask, 'TITLE'].apply(get_city_via_llm)

Error for [경기도박물관] 2026 문화동호회 신규회원 모집 (2/22~3/5): Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
Error for [처인성역사교육관] 겨울방학 교육프로그램 신청 (2/9~2/28): Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
Error for [경기도박물관] 특별전_성파선예: 성파스님의 예술세계 (2/10~5/31): Error code: 401 - {'error': {'message': 'Incorrect API key provided: your_ope************here. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}
Error for [한국민속촌] 설날 세시행사 (2/14~3/3): Error code: 401 - {'error': {'message': 'Incorrect API key provided

💾 Step 4. DBeaver 적재를 위한 마지막 처리
LLM이 생성한 CITY 컬럼을 포함하여 다시 CSV로 저장합니다.

In [24]:
# 최종 결과 저장
event_df.to_csv("gyeonggi_events_llm.csv", index=False, encoding="utf-8-sig")

### Step 1. 파이썬 데이터를 CSV 파일로 내보내기 (Export)
DBeaver의 '데이터 가져오기' 기능을 사용하려면 파이썬에 있는 데이터프레임을 먼저 엑셀(CSV) 파일로 저장해야 합니다. 앞서 작업하신 주피터 노트북 맨 아래에 이 코드를 추가해서 실행해 주세요.

### Step 2. DBeaver에서 데이터를 담을 '테이블(Table)' 만들기
데이터를 넣기 전, DBeaver에 빈 상자(테이블)가 준비되어 있어야 합니다. 건동님이 생성해주신 DB(예: fms 또는 public 스키마)에 SQL 편집기를 열고 아래 쿼리를 실행해 빈 테이블을 만듭니다.

In [17]:
import pandas as pd
from IPython.display import display

# 1. 업로드한 CSV 파일을 새로 읽어옵니다. 
# 파일 경로가 정확한지 확인해 주세요 (예: '경기도소식현황.csv')
file_path = "경기도소식현황.csv" 
event_df = pd.read_csv(file_path, encoding="utf-8-sig")

# 2. 도시 추출 로직 (한글 컬럼명 '제목', '기관명' 기준) 
def extract_city_improved(row):
    # 제목과 기관명을 합쳐서 검색 텍스트 생성
    search_text = f"{row['제목']} {row['기관명']}"
    
    # 용인 관련 키워드
    if any(k in search_text for k in ["용인", "민속촌", "처인", "기흥", "수지", "박물관"]):
        return "용인시"
    # 수원 관련 키워드
    elif any(k in search_text for k in ["수원", "화성행궁", "광교", "장안", "권선", "팔달", "영통"]):
        return "수원시"
    else:
        return "기타 경기도"

# 3. 'CITY' 파생 변수 생성 및 필터링
event_df['CITY'] = event_df.apply(extract_city_improved, axis=1)
target_city_df = event_df[event_df['CITY'].isin(['용인시', '수원시'])].copy()

# 4. 결과 출력
print(f"✅ 필터링 완료! 용인/수원 관련 데이터 총 {len(target_city_df)}건을 찾았습니다.")
display(target_city_df.head(20))

# 5. DBeaver용 CSV로 다시 저장
target_city_df.to_csv("gyeonggi_events_final.csv", index=False, encoding="utf-8-sig")
print("📂 'gyeonggi_events_final.csv' 파일로 저장되었습니다. 이제 DBeaver에 넣으시면 됩니다!")

FileNotFoundError: [Errno 2] No such file or directory: '경기도소식현황.csv'